In [6]:
from jobflow import Job, Flow
from jobflow_remote import submit_flow, set_run_config
from autoplex.auto.GenMLFF.jobs import RSS, evaluate_mlip_ensemble, QEscf

/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/nequip/__init__.py:20: UserWarning: !! PyTorch version 2.2.1+cu121 found. Upstream issues in PyTorch versions 1.13.* and 2.* have been seen to cause unusual performance degredations on some CUDA systems that become worse over time; see https://github.com/mir-group/nequip/discussions/311. The best tested PyTorch version to use with CUDA devices is 1.11; while using other versions if you observe this problem, an unexpected lack of this problem, or other strange behavior, please post in the linked GitHub issue.
  warnings.warn(


In [7]:
#Define RSS test parameters
rss_test_params = {
    "tag": "C", #Tag of systems. It can also be used for setting up elements and stoichiometry.
    "generated_struct_numbers": [3, 2], #Expected number of generated randomized unit cells for each run
    "buildcell_options": [ #Buildcell params for each buildcell run
        {"NFORM": '1', "SYMMOPS": "1-4", "SLACK": 0.25, "OVERLAP" : 0.1, "NATOM" : '{6,8,10,12,14,16,18,20,22,24}'}, 
        {"NFORM": '1', "SYMMOPS": "1-4", "SLACK": 0.25, "OVERLAP" : 0.1, "NATOM" : '{7,9,11,13,15,17,19,21,23}'}
    ],
    "fragment_file": None, #Fragment(s) for random structures, e.g. molecules, to be placed indivudally intact.
    "remove_tmp_files": True, #Remove all temporary files raised by buildcell to save memory
    "num_processes": 1, #Number of processes to use for parallel computation
}

In [8]:
#Define parameters for the ensemble evaluator: use pre-trained model as pilot
mlip_ensemble_params = {
    "mlip_type": "MACE",
    "mlip_paths": [
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/3A/MACE_stagetwo.model",
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/4A/small/MACE_stagetwo.model",
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/4A/ultra_small/MACE_stagetwo.model",
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/5A/MACE_stagetwo.model",
        ],
    "mlip_errors": [151.3, 125.6, 153.7, 130.8],
    "mlip_kwargs": {
        "device" : "cuda", 
        "default_dtype" : "float32",
        },
    "pre_trained_model": "/leonardo_work/EUHPC_A04_113/Alberto/mace/pre-trained-models/mace-mpa-0-medium.model",
    "pre_trained_kwargs": {
        "device" : "cuda", 
        "default_dtype" : "float32",
    },
}

In [9]:
#Define QE test parameters
qe_test_params = {
    "qe_run_cmd": "mpirun -np 1 pw.x",
    "num_qe_workers": 1, #Number of workers to use for the calculations. If None setp up 1 worker per scf
    "fname_pwi_template": "/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/Test-workflow/rss-mlip/reference.pwi", #Path to file containing the template QE input
}

In [10]:
#Define resources
parallel_cpu_resources = {
    "account": "IscrB_MLSilDia",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 32,
    "cpus_per_task": 1,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_cpu_resources = {
    "account": "IscrB_MLSilDia",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 32,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_gpu_resources = {
    "account": "IscrB_MLSilDia", 
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 8,
    "gres": "gpu:1",
    "mem": "120000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
    }

In [11]:
#Define the RSS job
randomized_structures_paths = RSS(**rss_test_params)

#Define the ensemble evaluator job
evaluate_mlip_ensemble = evaluate_mlip_ensemble(**mlip_ensemble_params, structure_paths=randomized_structures_paths.output)

#Define the QEscf job
qe_scf_job = QEscf(**qe_test_params, fname_structures=evaluate_mlip_ensemble.output)

In [12]:
# Define the wrokflow
rss_ensemble_qe_flow = Flow(jobs=[randomized_structures_paths, evaluate_mlip_ensemble, qe_scf_job], name="rss_ensemble_qe_flow")

In [13]:
rss_ensemble_qe_flow = set_run_config(
    rss_ensemble_qe_flow, name_filter="RSS", worker="mlff_relax_local", exec_config="rss_config", resources=serial_cpu_resources
)

rss_ensemble_qe_flow = set_run_config(
    rss_ensemble_qe_flow, name_filter="evaluate_mlip_ensemble", worker="mlff_mace", resources=serial_gpu_resources
)

rss_ensemble_qe_flow = set_run_config(
    rss_ensemble_qe_flow, name_filter="run_qe_worker", exec_config="qe_config", worker="QuantumEspresso", resources=serial_gpu_resources
)

In [14]:
# Append RSSautoplex-flow to jf jobs
submit_flow(
    rss_ensemble_qe_flow, worker="local_worker",
    resources={}, 
    project="GenMLFF",
)

2025-05-15 18:21:32,346 - INFO - Added flow (3641317f-c656-425f-8b4d-3ce7a8b63e48) with jobs: ('c2f9c6fb-34db-41ce-9f80-2151e07856d1', '91274e31-14bf-4f68-a4cc-49a326aa6c0f', '27766630-0ad7-4a81-a4cb-f7548266d7d0')


['4', '5', '6']